In [2]:
!pip install pandas transformers torch scikit-learn

In [4]:
import pandas as pd

from transformers import pipeline

In [7]:
df = pd.read_csv("support_tickets.csv")

In [8]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,119237,105834,True,Wed Oct 11 06:55:44 +0000 2017,@AppleSupport causing the reply to be disregar...,119236,NaN
1,119238,ChaseSupport,False,Wed Oct 11 13:25:49 +0000 2017,@105835 Your business means a lot to us. Pleas...,NaN,119239.0
2,119239,105835,True,Wed Oct 11 13:00:09 +0000 2017,@76328 I really hope you all change but I'm su...,119238,NaN
3,119240,VirginTrains,False,Tue Oct 10 15:16:08 +0000 2017,@105836 LiveChat is online at the moment - htt...,119241,119242.0
4,119241,105836,True,Tue Oct 10 15:17:21 +0000 2017,@VirginTrains see attached error message. I've...,119243,119240.0


In [9]:
df.columns

Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')

In [10]:
labels = [
    "Login Issue",
    "Payment",
    "Technical Problem",
    "Account",
    "Refund",
    "Delivery",
    "Bug Report",
    "Feature Request"
]

In [11]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

C:\Users\lenovo\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [12]:
ticket = df.iloc[0]["text"]

print(ticket)

@AppleSupport causing the reply to be disregarded and the tapped notification under the keyboard is opened😡😡😡


In [13]:
result = classifier(
    ticket,
    labels,
    multi_label=True
)

result

{'sequence': '@AppleSupport causing the reply to be disregarded and the tapped notification under the keyboard is opened😡😡😡',
 'labels': ['Technical Problem',
  'Bug Report',
  'Login Issue',
  'Refund',
  'Feature Request',
  'Payment',
  'Account',
  'Delivery'],
 'scores': [0.8878512382507324,
  0.21290196478366852,
  0.17443141341209412,
  0.16329774260520935,
  0.12351870536804199,
  0.08967004716396332,
  0.06321578472852707,
  0.0305668693035841]}

In [14]:
print("Top 3 Predicted Tags:")

for i in range(3):
    print(f"{result['labels'][i]} : {result['scores'][i]:.4f}")

Top 3 Predicted Tags:
Technical Problem : 0.8879
Bug Report : 0.2129
Login Issue : 0.1744


In [18]:
predictions = []

for ticket in df["text"][:100]:

    result = classifier(
        ticket,
        labels,
        multi_label=True
    )

    predictions.append(result["labels"][:3])

print("Completed!")

Completed!


In [ ]:
results = df.iloc[:100].copy()

results["Top_3_Tags"] = predictions

In [ ]:
results[["text", "Top_3_Tags"]].head()

In [ ]:
results.to_csv(
    "tagged_support_tickets.csv",
    index=False
)

print("File Saved Successfully!")